# Day12 pilot on Kaggle

Keep this notebook **private**, enable **Internet**, and select a **T4 GPU** accelerator in Kaggle settings. The code uses one GPU even if two are allocated. Run cells individually in order; the first cell only checks the runtime and creates fresh staging directories.

This transfers the fixed experiment from GitHub commit `eeaa0f946317f4313c76063dc17034fc0f1132f2`. It reuses the exact preserved guides; no translator retraining or guide preparation is needed. The actual 20-clip GPU comparison has not yet run, so this notebook does not promise a memory fit or a successful research result.

Public model downloads are about 19 GB. The FP32 text encoder also needs temporary offload space. Model weights and offload stay under `/tmp/vjepa-day12`; only small prompt caches, runtime records, generated outputs and metrics go under `/kaggle/working/day12`. Save a Kaggle version with outputs and recover the results to GitHub before ending the session. Do not put GitHub credentials in this notebook.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
import json
import os
import platform
import shutil
import subprocess
import sys

# No downloads, installs or model inference in this cell.
scratch = Path('/tmp/vjepa-day12')
saved = Path('/kaggle/working/day12')
revision = 'eeaa0f946317f4313c76063dc17034fc0f1132f2'
if not Path('/kaggle/working').is_dir():
    raise RuntimeError('This notebook expects a Kaggle Notebook runtime.')
if scratch.exists() or saved.exists():
    raise RuntimeError('A Day12 directory already exists. Preserve it; this setup will not overwrite it.')
env = dict(os.environ, PYTHONDONTWRITEBYTECODE='1')
gpu_check = """
import json, torch
import torch.nn.functional as F
assert torch.cuda.is_available(), 'Select a GPU accelerator before continuing'
assert hasattr(F, 'scaled_dot_product_attention'), 'Native PyTorch SDPA is required'
free, total = torch.cuda.mem_get_info(0)
print(json.dumps(dict(torch=torch.__version__, cuda=torch.version.cuda,
    gpu_count=torch.cuda.device_count(), gpu0=torch.cuda.get_device_name(0),
    capability=list(torch.cuda.get_device_capability(0)),
    gpu0_free_bytes=free, gpu0_total_bytes=total)))
"""
gpu = json.loads(subprocess.run(
    [sys.executable, '-c', gpu_check], check=True, text=True,
    capture_output=True, env=env).stdout)
nvidia_smi = subprocess.run(
    ['nvidia-smi'], check=True, text=True, capture_output=True).stdout
memory = {}
for line in Path('/proc/meminfo').read_text().splitlines():
    key, value = line.split(':', 1)
    if key in ('MemTotal', 'MemAvailable'):
        memory[key + '_bytes'] = int(value.split()[0]) * 1024
scratch_disk = shutil.disk_usage('/tmp')
saved_disk = shutil.disk_usage('/kaggle/working')
if scratch_disk.free < 40 * 1024**3:
    raise RuntimeError('Require at least 40 GiB free temporary disk for weights, offload and loading.')
report = dict(
    checked_utc=datetime.now(timezone.utc).isoformat(), checkpoint=revision,
    python=platform.python_version(), gpu=gpu, memory=memory,
    temporary_disk=dict(total=scratch_disk.total, free=scratch_disk.free),
    output_disk=dict(total=saved_disk.total, free=saved_disk.free),
    temporary_and_output_share_filesystem=(
        os.stat('/tmp').st_dev == os.stat('/kaggle/working').st_dev),
    note='GPU availability is checked; model memory fit remains untested.')
scratch.mkdir()
saved.mkdir()
(saved / 'preflight.json').write_text(json.dumps(report, indent=2) + '\n')
(saved / 'nvidia-smi.txt').write_text(nvidia_smi)
repo = scratch / 'repo'
models = scratch / 'model-cache'
guides = scratch / 'guides'
prompt_staging = scratch / 'prompt-staging'
prompts = saved / 'prompts'
renders = saved / 'render'
assessment = saved / 'assessment.json'
print(json.dumps(report, indent=2))
print('Preflight passed. Continue with the checkout cell; no model has been downloaded.')


{
  "checked_utc": "2026-10-10T22:55:30.090289+00:00",
  "checkpoint": "eeaa0f946317f4313c76063dc17034fc0f1132f2",
  "python": "3.13.15",
  "gpu": {
    "torch": "2.11.0+cu128",
    "cuda": "12.8",
    "gpu_count": 2,
    "gpu0": "Tesla T4",
    "capability": [
      7,
      5
    ],
    "gpu0_free_bytes": 15526068224,
    "gpu0_total_bytes": 15636037632
  },
  "memory": {
    "MemTotal_bytes": 33658318848,
    "MemAvailable_bytes": 31995518976
  },
  "temporary_disk": {
    "total": 8758385086464,
    "free": 1148790763520
  },
  "output_disk": {
    "total": 20957446144,
    "free": 20940587008
  },
  "temporary_and_output_share_filesystem": false,
  "note": "GPU availability is checked; model memory fit remains untested."
}
Preflight passed. Continue with the checkout cell; no model has been downloaded.


In [2]:
# Fetch only the experiment, preserved evidence and two Day11 dependencies.
if repo.exists():
    raise RuntimeError('Checkout already exists; nothing will be overwritten.')
subprocess.run([
    'git', 'clone', '--no-checkout', '--filter=blob:none', '--single-branch',
    '--branch', 'experiment/part-consistency-colab',
    'https://github.com/skbdps/vjepa-poc.git', str(repo)], check=True, env=env)
subprocess.run([
    'git', 'sparse-checkout', 'set', '--no-cone',
    '/experiments/day11/requirements.txt', '/experiments/day11/vace_bridge.py',
    '/experiments/day12/'], cwd=repo, check=True, env=env)
subprocess.run(['git', 'checkout', '--detach', revision], cwd=repo, check=True, env=env)
head = subprocess.run(
    ['git', 'rev-parse', 'HEAD'], cwd=repo, check=True, text=True,
    capture_output=True).stdout.strip()
changes = subprocess.run(
    ['git', 'status', '--porcelain'], cwd=repo, check=True, text=True,
    capture_output=True).stdout.strip()
if head != revision or changes:
    raise RuntimeError('The checkout does not match the fixed clean checkpoint.')
(saved / 'source_checkpoint.json').write_text(json.dumps(
    dict(repository='https://github.com/skbdps/vjepa-poc',
         commit=head, clean=True), indent=2) + '\n')

def run(*arguments):
    return subprocess.run(
        [sys.executable, *map(str, arguments)], cwd=repo, check=True, env=env)

print('Verified checkpoint:', head)


Cloning into '/tmp/vjepa-day12/repo'...
HEAD is now at eeaa0f9 Record successful Colab sign-in and GPU quota block; fix pilot launch instructions


Verified checkpoint: eeaa0f946317f4313c76063dc17034fc0f1132f2


In [3]:
# Preserve Kaggle's existing CUDA PyTorch; requirements do not install torch.
run('-m', 'pip', 'install', '--no-cache-dir', '--disable-pip-version-check',
    '-r', 'experiments/day11/requirements.txt')
dependency_check = """
import json, sys, torch
from importlib.metadata import version
from pathlib import Path
from diffusers import (WanVACEPipeline, WanVACETransformer3DModel, AutoencoderKLWan)
from transformers import AutoTokenizer, UMT5EncoderModel
assert torch.cuda.is_available(), 'CUDA became unavailable'
assert version('diffusers') == '0.35.1'
assert version('transformers') == '4.57.6'
names = ['torch','diffusers','transformers','huggingface_hub','accelerate',
         'safetensors','ftfy','numpy','Pillow']
report = {name: version(name) for name in names}
with Path(sys.argv[1]).open('x') as handle:
    json.dump(report, handle, indent=2)
print(json.dumps(report, indent=2))
"""
run('-c', dependency_check, saved / 'dependencies.json')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 72.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 218.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 357.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 239.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 265.6 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.29.0
    Uninstalling huggingface_hub-1.29.0:
      Successfully uninstalled huggingface_hub-1.29.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.23.1
    Uninstalling tokenizers-0.23.1:
      Successfully uninstalled tokenizers-0.23.1
  Attempting uninstall: diffusers
    Found existing installation: diffusers 0.40.0
    Uninstalling diffusers-0.40.0:
      Successfully uninstalled diffusers-0

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


{
  "torch": "2.11.0+cu128",
  "diffusers": "0.35.1",
  "transformers": "4.57.6",
  "huggingface_hub": "0.36.2",
  "accelerate": "1.14.0",
  "safetensors": "0.8.0",
  "ftfy": "6.3.1",
  "numpy": "2.1.3",
  "Pillow": "12.3.0"
}


CompletedProcess(args=['/usr/bin/python3', '-c', "\nimport json, sys, torch\nfrom importlib.metadata import version\nfrom pathlib import Path\nfrom diffusers import (WanVACEPipeline, WanVACETransformer3DModel, AutoencoderKLWan)\nfrom transformers import AutoTokenizer, UMT5EncoderModel\nassert torch.cuda.is_available(), 'CUDA became unavailable'\nassert version('diffusers') == '0.35.1'\nassert version('transformers') == '4.57.6'\nnames = ['torch','diffusers','transformers','huggingface_hub','accelerate',\n         'safetensors','ftfy','numpy','Pillow']\nreport = {name: version(name) for name in names}\nwith Path(sys.argv[1]).open('x') as handle:\n    json.dump(report, handle, indent=2)\nprint(json.dumps(report, indent=2))\n", '/kaggle/working/day12/dependencies.json'], returncode=0)

In [4]:
# Restore exact input arrays and independently validate every recorded hash.
if guides.exists():
    raise RuntimeError('Guide destination already exists; preserve it.')
run('experiments/day12/archive_evidence.py', 'unpack',
    '--source', 'experiments/day12/run_recovered/evidence/guides',
    '--out', guides)
guide_check = """
import json, sys
from pathlib import Path
sys.path.insert(0, sys.argv[1])
from common import load_guides, sha256
manifest, bundles = load_guides(sys.argv[2])
report = dict(complete=True, scenes=sorted(bundles),
    manifest_sha256=sha256(Path(sys.argv[2]) / 'manifest.json'),
    arrays_validated=True, input_kind='preserved_development_guides')
with Path(sys.argv[3]).open('x') as handle:
    json.dump(report, handle, indent=2)
print(json.dumps(report, indent=2))
"""
run('-c', guide_check, repo / 'experiments/day12', guides,
    saved / 'guide_restore.json')


{"complete": true, "files": 3, "kind": "guides"}
{
  "complete": true,
  "scenes": [
    14000,
    14001
  ],
  "manifest_sha256": "01e306cf57cac911bb4d8a7eb415182a60f433eda236fbd431fc174856386829",
  "arrays_validated": true,
  "input_kind": "preserved_development_guides"
}


CompletedProcess(args=['/usr/bin/python3', '-c', "\nimport json, sys\nfrom pathlib import Path\nsys.path.insert(0, sys.argv[1])\nfrom common import load_guides, sha256\nmanifest, bundles = load_guides(sys.argv[2])\nreport = dict(complete=True, scenes=sorted(bundles),\n    manifest_sha256=sha256(Path(sys.argv[2]) / 'manifest.json'),\n    arrays_validated=True, input_kind='preserved_development_guides')\nwith Path(sys.argv[3]).open('x') as handle:\n    json.dump(report, handle, indent=2)\nprint(json.dumps(report, indent=2))\n", '/tmp/vjepa-day12/repo/experiments/day12', '/tmp/vjepa-day12/guides', '/kaggle/working/day12/guide_restore.json'], returncode=0)

In [5]:
# GPU text encoding runs in its own process and exits before video inference.
# Temporary FP32 disk offload must not be included in persisted Kaggle output.
if prompt_staging.exists() or prompts.exists():
    raise RuntimeError('Prompt staging or saved prompts already exist; preserve them.')
run('experiments/day12/cache_prompts.py',
    '--out', prompt_staging, '--model-cache', models)
metadata = json.loads((prompt_staging / 'metadata.json').read_text())
if metadata.get('complete') is not True:
    raise RuntimeError('Prompt encoding did not complete.')
cache_files = ('metadata.json', 'prompt_14000.pt', 'prompt_14001.pt')
if not all((prompt_staging / name).is_file() for name in cache_files):
    raise RuntimeError('A required prompt cache file is missing.')
prompts.mkdir()
for name in cache_files:
    shutil.copyfile(prompt_staging / name, prompts / name)
# Offload files remain in temporary staging, outside the saved output directory.
print('Saved two prompt caches and metadata:', prompts)
print('Temporary disk free (GiB):', round(shutil.disk_usage(scratch).free / 1024**3, 2))


Fetching 9 files: 100%|██████████| 9/9 [00:44<00:00,  4.94s/it]
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 3/3 [00:11<00:00,  3.69s/it]
/usr/local/lib/python3.13/dist-packages/accelerate/utils/modeling.py:1615: UserWarning: The following device_map keys do not match any submodules in the model: ['encoder.embed_tokens']
  warnings.warn(


COMPLETE /tmp/vjepa-day12/prompt-staging/metadata.json
Saved two prompt caches and metadata: /kaggle/working/day12/prompts
Temporary disk free (GiB): 1059.01


In [ ]:
# Actual fixed comparison: 2 scenes x 2 noise seeds x 5 conditions.
# If interrupted, change only RESUME_RENDER to True and rerun THIS cell
# in the same runtime. The driver verifies exact source/input/runtime bindings.
RESUME_RENDER = False
if renders.exists() and not RESUME_RENDER:
    raise RuntimeError('Results already exist. Preserve them; use verified resume if appropriate.')
if RESUME_RENDER and not (renders / 'metrics.json').is_file():
    raise RuntimeError('No render checkpoint exists to resume.')
arguments = [
    'experiments/day12/render.py', '--guides', guides, '--prompt-dir', prompts,
    '--model-cache', models, '--out', renders]
if RESUME_RENDER:
    arguments.append('--resume')
run(*arguments)
state = json.loads((renders / 'metrics.json').read_text())
if state.get('complete') is not True or len(state.get('renders', {})) != 20:
    raise RuntimeError('The 20-clip comparison is not complete; inspect saved errors and metrics.')
print('All 20 clips completed. Continue with assessment.')


Loading checkpoint shards: 100%|██████████| 2/2 [00:06<00:00,  3.31s/it]


RENDER dev_14000/noise_1111/source_static
RENDER dev_14000/noise_1111/jepa
RENDER dev_14000/noise_1111/no_f
RENDER dev_14000/noise_1111/wrong_direction


In [ ]:
# Assessment refuses incomplete or changed render evidence.
if assessment.exists():
    raise RuntimeError('An assessment already exists; it will not be overwritten.')
run('experiments/day12/assess.py',
    '--render-root', renders, '--guide-root', guides, '--output', assessment)
assessment_result = json.loads(assessment.read_text())
if (assessment_result.get('complete') is not True
        or assessment_result.get('problems') != []
        or len(assessment_result.get('scores', {})) != 20):
    raise RuntimeError('Assessment is incomplete or invalid; preserve outputs and inspect assessment.json.')
print('Results:', renders)
print('Assessment:', assessment)
print('Save a private Kaggle version with outputs, then recover these results to GitHub.')


Do not describe the test as completed unless the renderer records all 20 clips and assessment succeeds. Preserve errors and partial outputs if allocation, memory or connectivity interrupts the run. The saved preflight records the actual Kaggle environment; a different runtime is a new experiment unless all resume bindings match.

The source checkpoint contains recovered original guide bytes and reconstructed continuation drivers. This is a framewise JEPA-guided video experiment, not a trained temporal JEPA model. Interpret the matched `jepa` versus `no_f` comparison before claiming a learned-translator advantage.


In [ ]:
# Notebook export cell; run only after the Day12 assessment has finished.
# Uses the paths defined by experiments/day12/kaggle_pilot.ipynb.
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import sys
import zipfile
from IPython.display import FileLink, display

saved = Path('/kaggle/working/day12')
repo = Path('/tmp/vjepa-day12/repo')
guides = Path('/tmp/vjepa-day12/guides')
sys.path.insert(0, str(repo / 'experiments/day12'))
from archive_evidence import child, sha256
from common import array_sha256, load_guides
from config import ARMS, NOISE_SEEDS, SCENES

render_root = saved / 'render'
export_assessment = json.loads((saved / 'assessment.json').read_text())
rendering = json.loads((render_root / 'metrics.json').read_text())
expected = {f'dev_{s}/noise_{n}/{a}' for s in SCENES for n in NOISE_SEEDS for a in ARMS}
assert len(expected) == 20
assert export_assessment['complete'] is True and export_assessment['problems'] == []
assert set(export_assessment['scores']) == expected and len(export_assessment['paired']) == 16
assert set(export_assessment['aggregate']) == set(ARMS)
assert rendering['complete'] is True and set(rendering['renders']) == expected
assert export_assessment['render_manifest_sha256'] == sha256(render_root / 'metrics.json')
assert export_assessment['assess_source_sha256'] == sha256(repo / 'experiments/day12/assess.py')
_, bundles = load_guides(guides)
guide_hash = sha256(guides / 'manifest.json')
assert export_assessment['guide_manifest_sha256'] == rendering['bindings']['guide_manifest_sha256'] == guide_hash
for name, digest in rendering['bindings']['source_files'].items():
    assert sha256(child(repo, name)) == digest
assert rendering['bindings']['prompt_metadata_sha256'] == sha256(saved / 'prompts/metadata.json')
payload_count = 0
for key, row in rendering['renders'].items():
    assert key == f"dev_{row['scene_seed']}/noise_{row['noise_seed']}/{row['arm']}"
    assert row['guide_array_sha256'] == array_sha256(bundles[row['scene_seed']]['guide_' + row['arm']])
    assert set(row['files']) == {'rgb', 'latent', 'gif'}
    for record in row['files'].values():
        path = child(render_root, record['path'])
        assert sha256(path) == record['sha256'] and path.stat().st_size == record['bytes']
        payload_count += 1
assert payload_count == 60

# Keep weights/offload out of the result archive; the notebook places them in /tmp.
blocked = {'models', 'model-cache', 'offload', 'prompt-staging'}
paths, excluded = [], []
for path in sorted(saved.rglob('*')):
    if not path.is_file():
        continue
    relative = path.relative_to(saved)
    if blocked.intersection(relative.parts) or path.suffix in {'.safetensors', '.bin'}:
        excluded.append(str(relative))
        continue
    child(saved, str(relative))  # Reject links escaping the result root.
    paths.append(path)
records = [{'path': str(p.relative_to(saved)), 'bytes': p.stat().st_size, 'sha256': sha256(p)} for p in paths]
manifest = {'version': 'day12_actual_video_export_v1', 'created_utc': datetime.now(timezone.utc).isoformat(),
            'assessment_complete': True, 'scores': 20, 'render_payloads_verified': 60,
            'guide_provenance_verified': True, 'files': records, 'excluded': excluded}
archive = saved.parent / 'day12_results.zip'
assert not archive.exists(), 'Preserve the existing archive; choose a new explicit filename if needed.'
with zipfile.ZipFile(archive, 'x', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    for path, record in zip(paths, records):
        z.write(path, record['path'])
    z.writestr('_export_manifest.json', json.dumps(manifest, indent=2) + '\n')
with zipfile.ZipFile(archive) as z:
    assert set(z.namelist()) == {r['path'] for r in records} | {'_export_manifest.json'}
    for record in records:
        data = z.read(record['path'])
        assert len(data) == record['bytes'] and hashlib.sha256(data).hexdigest() == record['sha256']
print(json.dumps({'archive': str(archive), 'bytes': archive.stat().st_size, 'sha256': sha256(archive),
                  'files': len(records), 'render_payloads_verified': payload_count,
                  'scores': len(export_assessment['scores']), 'problems': export_assessment['problems']}, indent=2))
display(FileLink(str(archive)))
